In [1]:
!pip install -U sentence-transformers pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_pickle("processed_documents.pkl")

print(df.head())
print(df.shape)

        doc_id                                      original_text  \
0  NCT00000102  Title: Congenital Adrenal Hyperplasia: Calcium...   
1  NCT00000104  Title: Does Lead Burden Alter Neuropsychologic...   
2  NCT00000105  Title: Vaccination With Tetanus and KLH to Ass...   
3  NCT00000106  Title: 41.8 Degree Centigrade Whole Body Hyper...   
4  NCT00000107  Title: Body Water Content in Cyanotic Congenit...   

                                        cleaned_text  
0  titl congenit adren hyperplasia calcium channe...  
1  titl lead burden alter neuropsycholog develop ...  
2  titl vaccin tetanu klh assess immun respons co...  
3  titl degre centigrad whole bodi hyperthermia t...  
4  titl bodi water content cyanot congenit heart ...  
(375580, 3)


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully")

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully


In [5]:
sample_docs = df["cleaned_text"].astype(str).tolist()[:50]

print("Number of docs:", len(sample_docs))
print(sample_docs[:3])

Number of docs: 50
['titl congenit adren hyperplasia calcium channel therapeut target condit summari studi test abil extend releas nifedipin procardia xl blood pressur medic permit decreas dose glucocorticoid medic children take treat congenit adren hyperplasia cah detail descript protocol design assess acut chronic effect calcium channel antagonist nifedipin axi patient congenit adren hyperplasia multicent trial compos two phase involv parallel design goal phase examin abil nifedipin placebo decreas adrenocorticotrop hormon acth level well begin assess nifedipin effect goal phase ii evalu effect nifedipin attenu acth releas nifedipin permit decreas dosag glucocorticoid need suppress hpa axi decreas would turn reduc deleteri effect glucocorticoid treatment cah', 'titl lead burden alter neuropsycholog develop condit summari inner citi children increas risk lead overburden turn affect cognit function howev underli neuropsycholog effect lead overburden effect well delin studi part larger 

In [6]:
def preprocess_query(query):
    return query.lower().strip()

In [7]:
def search(query, docs, doc_embeddings, model):
    
    query = preprocess_query(query)
    query_vec = model.encode([query])
    
    scores = cosine_similarity(query_vec, doc_embeddings)[0]
    top_idx = scores.argsort()[::-1]
    
    results = []
    
    for i in top_idx[:5]:
        results.append({
            "document": docs[i],
            "score": float(scores[i])
        })
    
    return results

In [9]:
sample_docs = df["cleaned_text"].astype(str).tolist()[:50]

sample_embeddings = model.encode(sample_docs, show_progress_bar=True)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
results = search(
    "invest in stock market",
    sample_docs,
    sample_embeddings,
    model
)

for r in results:
    print(r["score"])
    print(r["document"])
    print("------")

0.08763432502746582
titl diabet retinopathi vitrectomi studi drv condit summari compar two therapi earli vitrectomi convent manag recent sever vitreou hemorrhag secondari diabet retinopathi convent manag includ vitrectomi hemorrhag fail clear wait period 6 12 month retin detach involv center macula develop time compar earli vitrectomi convent manag eye good vision poor prognosi threaten hemorrhag retin detach sever prolif retinopathi studi natur histori sever prolif diabet retinopathi detail descript vitrectomi may remov vitreou hemorrhag also prevent reliev traction retina contract fibrovascular membran character sever prolif diabet retinopathi import determin whether earli intervent vitrectomi better visual outcom instead produc rate seriou complic higher rate associ convent manag two random trial carri drv among patient age 18 70 year either diabet first trial 616 patient recruit sever visual loss recent sever vitreou hemorrhag least one eye elig eye randomli assign either earli vit

In [12]:
def final_search(query):
    
    query_vec = model.encode([query])
    scores = cosine_similarity(query_vec, all_embeddings)[0]
    
    top_idx = scores.argsort()[::-1]
    
    for i in top_idx[:5]:
        print(scores[i])
        print(all_docs[i])
        print("-----")

In [14]:
df = pd.read_pickle("processed_documents.pkl")
all_docs = df["cleaned_text"].astype(str).tolist()

In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

all_embeddings = model.encode(
    all_docs,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/11737 [00:00<?, ?it/s]

In [17]:
print(all_embeddings.shape)

(375580, 384)


In [18]:
import numpy as np

np.save(
    "medical_embeddings.npy",
    all_embeddings
)

print("Embeddings Saved Successfully")

Embeddings Saved Successfully


In [19]:
loaded_embeddings = np.load("medical_embeddings.npy")

print(loaded_embeddings.shape)

(375580, 384)
